# Prévision des séries temporelles

Contenu

- Description de la tâche
- Un premier regard sur les données
- Analyse Exploratoire
- Analyse de corrélation
- Partage des données (Split)
- "Entraînement" (Training)
- Prédiction et évaluation

## Description de la tâche
 
Nous voulons développer un modèle pour prévoir la charge électrique de l'heure suivante sur la base de la charge électrique horaire et des données de température.

## Premier regard sur les données

Nous allons utiliser les données de la [Global Energy Forecasting Competition] (https://en.wikipedia.org/wiki/Global_Energy_Forecasting_Competition). Nous allons nous concentrer sur les données de l'année 2014 qui contiennent 8 760 observations. Le jeu de données a été téléchargé pour vous et est disponible dans le dossier *data*.


In [1]:
# Load settings and functions
%run ./scripts/tools.py

AttributeError: module 'matplotlib' has no attribute '__version_info__'

In [ ]:
df = load_data()

Let's have a close look at the dataset.

In [ ]:
info_data(df)

Examinons quelques lignes de l'ensemble de données.

In [ ]:
df.head(5)

## Analyse Exploratoire

Visualisons maintenant les charges électriques et la température. Notez qu'en raison du rôle central de la dimension temporelle dans les données de séries chronologiques, il convient d'explorer la dynamique des caractéristiques et de la cible au fil du temps (__tendances et cycles__). Il convient également de vérifier la corrélation entre la cible et ses propres valeurs dans le passé (__dépendances temporelles__). Enfin, il faut être attentif aux changements dans la distribution des données au fil du temps (__stationnarité__). Comme nos données sont stationnaires, nous ne vérifions que les tendances, les cycles et les dépendances temporelles dans ce qui suit.

In [ ]:
plot_data(df)

<div class="alert alert-success">
<h3>Questions</h3>
    
__Q1.__ À quoi ressemblent les distributions dans la première rangée de graphiques ?<br>
    
__Q2.__ Quelle est le trait commun entre la charge électrique et de la dynamique de la température dans les deuxième et troisième rangées de graphiques ?<br>
    
__Q3.__ En ce qui concerne la quatrième rangée, est-ce que la charge électrique reste constante au cours de la semaine ? Qu'en est-il de la nuit par rapport à la journée ?<br>
    
__Q4.__ Que vous apprend la carte thermique de la dernière ligne? Les mois de l'année ont-ils une incidence sur l'intensité de la charge électrique entre 6 heures et minuit?

 
 💡 Les réponses à ces questions permettent d'élaborer la stratégie de modélisation.

    
</div>

### Réponses

__Q1.__ <br>

__Q2.__ <br>

__Q3.__ <br>

__Q4.__ 

## Analyse de corrélation

Nous créons ici deux caractéristiques (feaures) qui peuvent jouer un rôle important dans la prévision de la charge. Une hypothèse simple est que **les valeurs passées de la charge et de la température peuvent prédire la charge**. Nos données ont une fréquence horaire et notre objectif est de prévoir la charge une heure à l'avance. Par conséquent, nous créons des décalages de la charge et de la température, puis nous vérifions les corrélations et les auto-corrélations.

In [ ]:
plot_corr(df, n_lags=1) # n_lag is between 1 and 24 hours

<div class="alert alert-info">
    
Modifions le nombre de décalages `n_lags` dans l'analyse de corrélation `plot_corr()`.<br>
    
 💡 **Indication:** Vous pouvez utiliser les raccourcis `C` et `V` (ou l'onglet `Editer`) pour placer une copie de la cellule juste en dessous de l'original et comparer ensuite deux valeurs différentes pour `n_lags`.<br>
    
</div>

<div class="alert alert-success">
<h3>Questions</h3>
    
__Q5.__ Si vous deviez prédire la charge actuelle sur la base du passé, laquelle des valeurs passées choisiriez-vous?<br>
    
__Q6.__ Constatez-vous une relation linéaire entre la charge actuelle et la charge d'il y a 1 heure ?<br>
    
__Q7.__ Observez-vous une relation linéaire entre la charge de courant et la température il y a 1 heure ?

    
 💡 Les réponses à ces questions permettent d'élaborer la stratégie de modélisation.
    
</div>

### Réponses

__Q5.__ <br>

__Q6.__ <br>

__Q7.__ 

## Partageons les données

Partageons l'ensemble des données. Nous utilisons le dernier mois des données, c'est-à-dire décembre, pour les tests et le reste des données, c'est-à-dire de janvier à novembre, pour l'entraînement et les validations.

In [ ]:
# Train/test splitting
train, test = sample_split(df)

## Entraînement ("Training")
Sur la base de l'analyse des corrélations, nous décidons d'utiliser les premiers décalages de la charge et de la température comme caractéristiques dans notre modèle. Nous savons également que ces caractéristiques sont liées à la cible de manière linéaire et non linéaire. Par conséquent, nous construisons des modèles linéaires et non linéaires pour prédire la charge électrique à une heure d'avance. 

Nous choisissons la régression ridge ou la forêt aléatoire comme modèle d'apprentissage automatique. Il convient de noter que chaque modèle nécessite un prétraitement approprié des caractéristiques. Par exemple, la régression ridge nécessite la mise à l'échelle des caractéristiques continues, et les deux modèles requièrent un codage à un instant des caractéristiques catégorielles.

In [ ]:
# Fit a ridge regression or a random forest model on the train data
model = train_model(train, select_model='regression') # select_model = 'regression' or 'randomforest'

<div class="alert alert-success">
<h3>Questions</h3>
    
__Q8.__ Qu'est-ce que vous remarquez sur le graphique lorsque vous sélectionnez la régression ridge ? L'erreur reste-t-elle constante lorsque l'on modifie $\alpha$ ? <br>
    
__Q9.__ Laquelle des courbes indique une meilleure performance (ou une erreur plus faible), l'entraînement ou la validation ? Qu'indique l'écart entre les deux ?
    
</div>

### Réponses

__Q8.__

__Q9.__

## Prédiction et évaluation
La première étape consiste à définir notre métrique d'évaluation et notre ligne de base. Nous choisissons la **moyenne des erreurs absolues (MAE) comme mesure** et **la médiane comme ligne de base**. Notez que vous pouvez même considérer les charges de l'heure précédente comme une prédiction de la charge actuelle sans aucune modélisation. En d'autres termes, **les charges décalées peuvent servir de référence (intelligente) avant de construire un modèle**. Ci-dessous, vous pouvez voir comment la performance d'une telle base peut être comparée à la base statistique et à nos modèles.

Evaluons la performance du modèle que nous avons précédemment choisi par `select_model` et entraînons-le.

In [ ]:
# evaluate the model performance  
# n_days shows the actual and predicted values for the number of days that you select
# n_days takes values between 1 and 31
evaluate_model(model, train, test, n_days=1) 

Le premier graphique montre que le modèle ne surpasse pas de manière remarquable la ligne de base intelligente comme il le fait pour la première ligne de base, c'est-à-dire la médiane. Il s'agit d'une situation courante pour les données de séries temporelles avec une forte auto-corrélation qui peut mettre les modèles d'apprentissage automatique dans l'analyse des séries temporelles dans une position difficile à justifier et à déployer. 

Dans le second graphique, nous montrons la charge prédite avec le modèle de votre choix dans `select_model` ainsi que les charges observées dans l'ensemble de test pour la période de `n_days`. Le graphique montre également les deux lignes de base.

Notez que notre objectif était de construire des modèles capables de prédire les charges électriques une heure à l'avance. Mais il est possible de développer une configuration permettant de modifier l'horizon de prédiction pour qu'il soit supérieur à une heure. Le choix de l'horizon de prédiction dépend du domaine, du problème à résoudre et de la valeur ajoutée du projet d'apprentissage automatique.

<div class="alert alert-info">

Let’s compare the ridge regression and the random forest model. <br>
    
1. Go back to the subsection **Training** and change the `select_model`-parameter inside the `train_model`-function to `'randomforest'` (Careful you need the quotation marks).<br>
    
2. Run `evaluate_model` again and compare the results.
    
 💡 **Tip:** You can use the shortcuts `C` and `V` (or the `Edit` tab) to place copies of the two elevant cells below this task to make the comparison of the performances easier.
</div>

<div class="alert alert-success">
<h3>Questions</h3>
    
__Q10.__ Quelle est l'idée derrière avoir une base de référence ("baseline")?<br>
    
__Q11.__ Laquelle des deux lignes de base utiliseriez-vous ? La régression ridge est-elle plus performante que les deux bases de référence?<br>
    
__Q12.__ Laquelle de la régression ridge ou de la forêt aléatoire est la plus performante sur les données de test ?

</div>

### Answers

__Q10.__ <br>

__Q11.__ <br>

__Q12.__ 